# Attention Pattern Analysis

This notebook demonstrates how to extract and analyze attention patterns to understand how models process reasoning tasks.

## What You'll Learn

1. Extracting attention patterns from model layers
2. Visualizing attention heatmaps and head-wise patterns
3. Computing attention rollout (cumulative attention flow)
4. Identifying important attention heads for reasoning
5. Analyzing attention entropy and distance

**Estimated time**: 15 minutes  
**Prerequisites**: Complete notebook 01_quickstart.ipynb

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
from interpretability import load_model
from interpretability.extraction import (
    extract_attention_patterns,
    compute_attention_rollout,
    get_head_importance,
    compute_attention_entropy,
    compute_attention_distance
)
from interpretability.visualization import (
    plot_attention_heatmap,
    plot_attention_heads_grid,
    plot_attention_rollout,
    plot_head_importance,
    plot_attention_entropy,
    plot_token_attention_evolution
)

print("✓ Imports successful")

## 1. Load Model and Prepare Input

In [ ]:
# Load DeepSeek model with platform-appropriate settings
from interpretability import get_recommended_device

device = get_recommended_device()
print(f"Using device: {device}")

model = load_model("deepseek-1.5b", device=device, quantization=None)
print(model)

In [ ]:
# Prepare a reasoning prompt
prompt = "What is 12 + 7? Let me think step by step. First, I'll add 10 + 7 = 17, then add 2 more: 17 + 2 = 19."

# Tokenize and move to device
inputs = model.tokenize(prompt, move_to_device=True)
tokens = model.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print(f"Input shape: {inputs['input_ids'].shape}")
print(f"Input device: {inputs['input_ids'].device}")
print(f"\nTokens ({len(tokens)}):")
for i, token in enumerate(tokens):
    print(f"  {i:2d}: {token}")

## 2. Extract Attention Patterns

Extract attention weights from all layers and heads.

In [ ]:
# Extract attention patterns
attention = extract_attention_patterns(
    model,
    inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    include_tokens=True
)

print(f"Attention patterns shape: {attention.patterns.shape}")
print(f"  Layers: {attention.num_layers}")
print(f"  Heads per layer: {attention.num_heads_per_layer[0]}")
print(f"  Sequence length: {attention.seq_len}")

## 3. Visualize Attention Heatmaps

Let's visualize attention patterns at different layers.

In [ ]:
# Visualize attention at layer 15 (mid-layer)
fig = plot_attention_heatmap(
    attention,
    tokens,
    layer=15,
    head=None,  # Aggregate across all heads
    title="Attention Pattern - Layer 15 (Mean Across Heads)"
)
fig.show()

In [ ]:
# Visualize attention at layer 25 (late layer)
fig = plot_attention_heatmap(
    attention,
    tokens,
    layer=25,
    head=None,
    title="Attention Pattern - Layer 25 (Mean Across Heads)"
)
fig.show()

## 4. Analyze Individual Attention Heads

Different attention heads specialize in different patterns. Let's look at all heads in a layer.

In [ ]:
# Grid view of all heads in layer 20
fig = plot_attention_heads_grid(
    attention,
    tokens,
    layer=20,
    max_heads=16
)
fig.show()

In [ ]:
# Look at a specific interesting head
fig = plot_attention_heatmap(
    attention,
    tokens,
    layer=20,
    head=3,  # Try different heads: 0-15
    title="Attention Pattern - Layer 20, Head 3"
)
fig.show()

## 5. Compute Attention Rollout

Attention rollout shows cumulative attention flow through all layers.

In [ ]:
# Compute rollout
rollout = compute_attention_rollout(
    attention,
    discard_ratio=0.1,
    head_fusion="mean"
)

print(f"Rollout shape: {rollout.shape}")

# Visualize
fig = plot_attention_rollout(
    rollout[0].cpu().numpy(),
    tokens,
    title="Cumulative Attention Flow (Rollout)"
)
fig.show()

## 6. Find Important Attention Heads

Which heads are most important for attending to the answer tokens?

In [ ]:
# Find tokens containing the answer ("19")
answer_indices = [i for i, token in enumerate(tokens) if '19' in token]
print(f"Answer token indices: {answer_indices}")
print(f"Answer tokens: {[tokens[i] for i in answer_indices]}")

In [ ]:
# Rank heads by importance to answer tokens
head_importance = get_head_importance(
    attention,
    target_tokens=answer_indices,
    method="mean"
)

print("Top 10 most important heads:")
print(head_importance.head(10))

In [ ]:
# Visualize head importance
fig = plot_head_importance(
    head_importance,
    top_k=20,
    title="Top 20 Attention Heads for Answer Tokens"
)
fig.show()

## 7. Attention Entropy Analysis

Entropy measures how focused or diffuse attention is.

In [ ]:
# Compute entropy at layer 20
entropy = compute_attention_entropy(attention)
layer_entropy = entropy[20, 0]  # [num_heads, seq_len]

print(f"Entropy shape for layer 20: {layer_entropy.shape}")
print(f"Mean entropy: {layer_entropy.mean().item():.3f} bits")
print(f"Min entropy: {layer_entropy.min().item():.3f} bits (most focused)")
print(f"Max entropy: {layer_entropy.max().item():.3f} bits (most diffuse)")

In [ ]:
# Visualize entropy
fig = plot_attention_entropy(
    layer_entropy.cpu().numpy(),
    tokens,
    layer=20,
    title="Attention Entropy - Layer 20"
)
fig.show()

print("\nInterpretation:")
print("  🟢 Green (low entropy) = Focused attention")
print("  🔴 Red (high entropy) = Diffuse attention")

## 8. Attention Distance

How far do tokens attend on average?

In [ ]:
# Compute attention distance
distance = compute_attention_distance(attention)
layer_distance = distance[20, 0]  # [num_heads, seq_len]

print(f"Distance shape for layer 20: {layer_distance.shape}")
print(f"Mean distance: {layer_distance.mean().item():.2f} tokens")

# Visualize
from interpretability.visualization import plot_attention_distance
fig = plot_attention_distance(
    layer_distance.cpu().numpy(),
    tokens,
    layer=20,
    title="Average Attention Distance - Layer 20"
)
fig.show()

print("\nInterpretation:")
print("  Lower values = Tokens attend to nearby tokens (local)")
print("  Higher values = Tokens attend to distant tokens (global)")

## 9. Track Token Attention Evolution

How does attention to a specific token evolve across layers?

In [ ]:
# Pick an interesting token (e.g., the answer)
if answer_indices:
    target_idx = answer_indices[0]
else:
    target_idx = len(tokens) - 1  # Last token

print(f"Tracking token: '{tokens[target_idx]}' (index {target_idx})")

# Plot attention TO this token across layers
fig = plot_token_attention_evolution(
    attention,
    token_idx=target_idx,
    direction="to"
)
fig.show()

In [ ]:
# Plot attention FROM this token across layers
fig = plot_token_attention_evolution(
    attention,
    token_idx=target_idx,
    direction="from"
)
fig.show()

## 10. Compare Different Layers

Let's compare early, middle, and late layers side by side.

In [ ]:
# Early layer (layer 5)
fig1 = plot_attention_heatmap(attention, tokens, layer=5, title="Early Layer (5)")
fig1.show()

# Middle layer (layer 15)
fig2 = plot_attention_heatmap(attention, tokens, layer=15, title="Middle Layer (15)")
fig2.show()

# Late layer (layer 25)
fig3 = plot_attention_heatmap(attention, tokens, layer=25, title="Late Layer (25)")
fig3.show()

print("\nObservations:")
print("  Early layers: Often show syntactic patterns (adjacent tokens)")
print("  Middle layers: Show semantic relationships and reasoning steps")
print("  Late layers: Focus on answer-relevant tokens")

## Summary

In this notebook, you learned how to:

1. ✅ Extract attention patterns from transformer layers
2. ✅ Visualize attention heatmaps for different layers and heads
3. ✅ Compute attention rollout to track cumulative attention flow
4. ✅ Identify important attention heads for specific tokens
5. ✅ Analyze attention entropy (focused vs diffuse)
6. ✅ Measure attention distance (local vs global)
7. ✅ Track how attention evolves across layers

## Key Insights

- **Different heads specialize**: Some focus on syntax, others on semantics
- **Layer hierarchy**: Early layers handle low-level patterns, late layers focus on task-specific information
- **Attention patterns reveal reasoning**: You can see which tokens the model considers when computing answers

## Next Steps

- **Notebook 03**: Circuit discovery with activation patching
- **Notebook 04**: Logit lens to track answer emergence
- Try different prompts and see how attention patterns change!

---

**Experiment Ideas**:
- Try multi-hop reasoning prompts
- Compare attention patterns for correct vs incorrect reasoning
- Find "reasoning heads" that consistently focus on intermediate steps